# Tesseract Maltese (mlt) OCR Test
This notebook loads images from `CROPS_DIR` path and writes transciption in a `.txt` file saved in `OUTPUT_DIR` using the Tesseract fine Tuned model.

## 1. Environment And Imports
This step imports all required libraries for file handling, OCR, and tabular analysis.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import urllib.request

import pandas as pd
import pytesseract
from PIL import Image

## 2. Dataset And OCR Configuration
Set workspace paths, select the JSON test split, point to image files, and set OCR language to `mlt`.

In [ ]:
CROPS_DIR = "path/to/your/crops_directory"  # Update this path to your actual crops directory
LANG = "mlt_custom_v1" # fine tuned tesseract language code
OUTPUT_DIR = "path/to/your/output_directory"  # Update this path to your actual output directory

TESSDATA_CUSTOM = "path/to/your/tesstrain_data/tessdata_custom"  # Update this path to your actual tessdata_custom directory

print(f"Crops dir exists: {CROPS_DIR.exists()} -> {CROPS_DIR}")

## 3. Verify Tesseract Language Data
Check whether the `mlt` traineddata is installed; if missing, download and configure it automatically.

In [ ]:
lang_output = subprocess.run(["tesseract", "--list-langs"], capture_output=True, text=True, check=True).stdout
available_langs = {line.strip() for line in lang_output.splitlines() if line.strip() and not line.startswith("List of available languages")}

if LANG not in available_langs:
    TESSDATA_CUSTOM.mkdir(exist_ok=True)
    traineddata_path = TESSDATA_CUSTOM / f"{LANG}.traineddata"
    if not traineddata_path.exists():
        url = f"https://github.com/tesseract-ocr/tessdata_best/raw/main/{LANG}.traineddata"
        urllib.request.urlretrieve(url, traineddata_path)
    os.environ["TESSDATA_PREFIX"] = str(TESSDATA_CUSTOM)
    print(f"Downloaded and configured {LANG}.traineddata in {TESSDATA_CUSTOM}")
else:
    print(f"Language '{LANG}' is already available in this environment.")

print("TESSDATA_PREFIX:", os.environ.get("TESSDATA_PREFIX", "<system default>"))

## 4. Transcibe the Prediction
Compute crop-level CER by comparing each OCR prediction.

In [ ]:
# predict on crops and save results under OUTPUT_DIR with matching names
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results = []

for crop_path in CROPS_DIR.rglob("*.png"):
    text = pytesseract.image_to_string(Image.open(crop_path), lang=LANG)
    rel_path = crop_path.relative_to(CROPS_DIR)
    out_txt_path = OUTPUT_DIR / rel_path.parent / f"tuned_{rel_path.stem}.txt"
    out_txt_path.parent.mkdir(parents=True, exist_ok=True)

    results.append({
        "crop": crop_path.name,
        "crop_path": str(crop_path),
        "output_txt": str(out_txt_path),
        "text": text,
    })

    with open(out_txt_path, "w", encoding="utf-8") as f:
        f.write(text)

results_df = pd.DataFrame(results)
print(f"Saved {len(results_df)} OCR text files under: {OUTPUT_DIR}")